# 01 — Data, provenance, and account-disjoint splits

This notebook is the entry point for the Czech-bank data used by PRAGMA-lite. The repository commits compact processed artifacts so readers can reproduce the model pipeline without downloading the raw Teradata export. See [the data card](docs/data.md) for provenance, scope, and licensing context.

The original Indian dataset was useful for practicing EDA, but the client histories were too sparse for an event-history encoder. The Czech source has about one million dated transactions and related account, client, card, loan, and district records, making it a practical small-scale proxy for the multi-source structure in PRAGMA.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Run this notebook from the repository root.')
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'czech_bank'
sorted(path.name for path in PROCESSED_DIR.glob('*.parquet'))

## What was kept, and why

`events_*.parquet` holds transaction rows. `profile_*.parquet` holds static profile values. `lifelong_events.parquet` holds dated account-opening, card-issuance, and loan-grant milestones. At sample time, the data handler materializes profile state at the sample cutoff; events after that date are not included. `loan_outcomes.parquet` is retained only for the exploratory loan diagnostic.

The split unit is **account ID**, never an individual transaction. This prevents the same account appearing in both train and test. Numeric bucket edges and categorical token vocabularies are fit from the train split only.

In [ ]:
manifest = pd.read_parquet(PROCESSED_DIR / 'account_split_manifest.parquet')
events = {split: pd.read_parquet(PROCESSED_DIR / f'events_{split}.parquet') for split in ('train', 'valid', 'test')}
profiles = {split: pd.read_parquet(PROCESSED_DIR / f'profile_{split}.parquet') for split in ('train', 'valid', 'test')}

assert manifest['account_id'].is_unique
assert set(manifest['split']) == {'train', 'valid', 'test'}
assert all(set(events[split]['account_id']) == set(profiles[split]['account_id']) for split in events)
{split: {'accounts': len(profiles[split]), 'events': len(events[split])} for split in events}

## EDA decisions that shape the model

The source has dates but not times of day. Events are therefore ordered by `(account_id, trans_date, trans_id)`: `trans_id` gives a deterministic within-day tie-break, but no artificial intraday timing is invented. Transaction histories can be long, so pre-training uses a maximum event context window rather than padding every account to the global maximum. The first full run uses a 256-event window.

There is no usable free-text description field, so this project uses a structured key/value tokenizer rather than a wordpiece tokenizer. Amount and balance are train-fitted quantile buckets; categorical fields map through a JSON vocabulary with explicit missing and unknown values.

## Optional: rebuild processed artifacts from raw data

This is intentionally off by default. The raw export remains local-only and is ignored by Git. The committed processed artifacts are enough for the later notebooks. Set `REBUILD=False` to `True` only after placing the raw TSVs in `financial_db_Teradata/`; use a staging output first if you want to compare versions.

In [ ]:
import subprocess
import sys

REBUILD = False
RAW_DIR = PROJECT_ROOT / 'financial_db_Teradata'
STAGING_DIR = PROJECT_ROOT / 'data' / 'processed' / 'czech_bank_rebuilt'
if REBUILD:
    subprocess.run([sys.executable, 'scripts/prepare_czech_data.py', '--raw-dir', str(RAW_DIR), '--output-dir', str(STAGING_DIR)], check=True)